# Phase 4: User Profile Modeling & Personalized Recommendation Engine

**Project**: Personalized Movie Recommendation System  
**Repository**: `Dp8453/Personalized-movie-recommendation-system`  
**Input Dataset**: `data/processed/clean_movies.csv` (4,803 movies)  

---  
### Objectives of Phase 4:
1. Transition from single-movie lookups (Phase 3) to **User-Centered Content-Based Personalization**.
2. Represent user watch histories with 1–5 integer ratings mapped to preference weights ($[-1.0, -0.5, 0.0, 0.5, 1.0]$).
3. Construct a weighted User Preference Profile vector: $u = \frac{\sum w_i v_i}{\sum |w_i|}$.
4. Compare the user profile against all 4,803 movie TF-IDF vectors using Cosine Similarity ($O(N)$ efficiency).
5. Strictly exclude previously rated movies from final recommendations.
6. Demonstrate negative preference impacts, edge case error handling, and the net-weight zero `[5, 1]` preference case.

## 1. Import Required Libraries & Modules

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Ensure src module can be imported from parent directory
sys.path.append(os.path.abspath('..'))
from src.personalizer import PersonalizedRecommender
from src.recommender import MovieRecommender

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
print('Imports successful.')

## 2. Load Processed Dataset (`data/processed/clean_movies.csv`)

In [ ]:
processed_path = os.path.join('..', 'data', 'processed', 'clean_movies.csv')
df = pd.read_csv(processed_path)
print(f'Processed Dataset Shape: {df.shape}')
print(f"Total Movies: {len(df)}")
assert len(df) == 4803, f'Expected 4,803 movies, got {len(df)}'
df[['id', 'title', 'tags']].head(3)

## 3. Instantiate PersonalizedRecommender (Reusing Phase 3 TF-IDF Space)

In [ ]:
personalizer = PersonalizedRecommender(df, max_features=5000, stop_words='english')
print(f"TF-IDF Feature Matrix Shape : {personalizer.tfidf_matrix.shape}")
print(f"Vocabulary Size             : {len(personalizer.vectorizer.vocabulary_)}")

## 4. Define User Preference History & Rating Transformation

### Rating-to-Preference Weight Mapping:
| Integer Rating | Meaning | Preference Weight ($w_i$) |
| :---: | :--- | :---: |
| **5** | Strongly Liked | **+1.0** |
| **4** | Liked | **+0.5** |
| **3** | Neutral | **0.0** |
| **2** | Disliked | **-0.5** |
| **1** | Strongly Disliked | **-1.0** |

### Rationale for Negative Preference Weights:
Treating all watched movies as positive signals introduces severe bias. Assigning negative weights to disliked movies (ratings 1 and 2) allows the user profile vector to be pushed away from unappealing themes while pulling it toward liked genres and actors.

In [ ]:
user_history = [
    ('Avatar', 5),
    ('Aliens', 5),
    ('The Dark Knight', 4),
    ('Titanic', 1)
]

print('User Rating History:')
validated = personalizer.validate_user_ratings(user_history)
for title, rating, idx, weight in validated:
    print(f"  Movie: {title:20s} | Rating: {rating} | Weight: {weight:+.1f}")

## 5. Build User Preference Profile Vector

Formally computed as:
$$u = \frac{\sum_i w_i v_i}{\sum_i |w_i|}$$
The profile vector shares the exact 5,000 feature dimension as movie TF-IDF vectors.

In [ ]:
user_profile = personalizer.build_user_profile(user_history)
print(f'User Profile Vector Shape: {user_profile.shape}')
print(f'Non-zero features in profile: {np.count_nonzero(user_profile)}')
print(f'Min value: {user_profile.min():.4f} | Max value: {user_profile.max():.4f}')

## 6. Generate Top-5 Personalized Movie Recommendations

In [ ]:
recs = personalizer.recommend_for_user(user_history, top_n=5)
print('===================================================')
print('TOP 5 PERSONALIZED RECOMMENDATIONS FOR USER')
print('===================================================')
for idx, r in enumerate(recs, 1):
    print(f" {idx}. {r['title']:35s} | Score: {r['personalized_score']:.4f}")

## 7. Compare Phase 3 (Single Movie) vs Phase 4 (Personalized User Profile)

In [ ]:
# Phase 3 Single Movie Recommendation for 'Avatar'
phase3_recs = personalizer.recommender.recommend('Avatar', top_n=5)

print('--- PHASE 3 RECOMMENDATIONS (Single Query Movie: Avatar) ---')
for idx, r in enumerate(phase3_recs, 1):
    print(f"  {idx}. {r['title']:35s} | Similarity: {r['similarity_score']:.4f}")

print('\n--- PHASE 4 PERSONALIZED RECOMMENDATIONS (User History: Avatar:5, Aliens:5, Dark Knight:4, Titanic:1) ---')
for idx, r in enumerate(recs, 1):
    print(f"  {idx}. {r['title']:35s} | Score: {r['personalized_score']:.4f}")

## 8. Demonstrate Negative Preference Impact & The `[5, 1]` Net Weight Zero Case

### Case A vs Case B:
- **Case A**: User likes both *Avatar* (5) AND *Titanic* (5).
- **Case B**: User likes *Avatar* (5) BUT strongly dislikes *Titanic* (1).

### The `[5, 1]` Net-Weight Zero Preference Case:
- Weights: $+1.0$ (Avatar) and $-1.0$ (Titanic).
- Net weight $= +1.0 + (-1.0) = 0.0$.
- $\sum |w_i| = 1.0 + |-1.0| = 2.0 \neq 0$.
- **Result**: The profile is 100% valid because positive and negative signals balance out to create a refined contrast vector!

In [ ]:
history_pos = [('Avatar', 5), ('Titanic', 5)]
history_neg = [('Avatar', 5), ('Titanic', 1)]

recs_pos = personalizer.recommend_for_user(history_pos, top_n=3)
recs_neg = personalizer.recommend_for_user(history_neg, top_n=3)

print('=== CASE A: Avatar=5, Titanic=5 (Positive Romance/Action Blend) ===')
for idx, r in enumerate(recs_pos, 1):
    print(f"  {idx}. {r['title']:35s} | Score: {r['personalized_score']:.4f}")

print('\n=== CASE B: Avatar=5, Titanic=1 (Net Weight = 0, Disliked Romance) ===')
for idx, r in enumerate(recs_neg, 1):
    print(f"  {idx}. {r['title']:35s} | Score: {r['personalized_score']:.4f}")

## 9. Error Handling & Edge Cases Verification

In [ ]:
# Edge Case 1: All-Neutral Ratings (sum(abs(w_i)) == 0)
try:
    personalizer.build_user_profile([('Avatar', 3), ('Titanic', 3)])
    print('FAIL: Expected ValueError for all-neutral history was not raised!')
except ValueError as e:
    print(f"SUCCESS 1: Caught all-neutral history error:\n  -> {e}")

# Edge Case 2: Duplicate Movie Titles in History
try:
    personalizer.build_user_profile([('Avatar', 5), ('avatar', 4)])
    print('FAIL: Expected ValueError for duplicate titles was not raised!')
except ValueError as e:
    print(f"\nSUCCESS 2: Caught duplicate title error:\n  -> {e}")

# Edge Case 3: Invalid Rating Values
try:
    personalizer.build_user_profile([('Avatar', 6)])
    print('FAIL: Expected ValueError for out-of-range rating was not raised!')
except ValueError as e:
    print(f"\nSUCCESS 3: Caught out-of-range rating error:\n  -> {e}")

# Edge Case 4: Non-existent Movie Title
try:
    personalizer.build_user_profile([('UnknownMovie999', 5)])
    print('FAIL: Expected ValueError for unknown movie was not raised!')
except ValueError as e:
    print(f"\nSUCCESS 4: Caught unknown movie error:\n  -> {e}")

## 10. Phase 4 Summary & Conclusions

### Complete Personalized Pipeline:
```
  [ User Rating History ] (e.g. Avatar: 5, Titanic: 1)
            │
            ▼  (Rating to Preference Weight: 1 -> -1.0, 5 -> +1.0)
  [ Preference Weights (w_i) ]
            │
            ▼  (Weighted Vector Sum / Sum of Absolute Weights)
  [ User Preference Profile Vector (1 x 5000) ]
            │
            ▼  (Cosine Similarity against all 4,803 Movie Vectors)
  [ Candidate Similarity Vector (1 x 4803) ]
            │
            ▼  (Exclude History Movies + Descending Sort)
  [ Top-N Ranked Personalized Movie Recommendations ]
```

- **Personalization Achieved**: Successfully implemented **Content-Based Personalization**.
- **Clarification**: This engine uses TF-IDF metadata vector profiles. It does **NOT** use Collaborative Filtering or deep learning models.
- **Next Phase**: Phase 5 will introduce offline model evaluation metrics (Precision@K, Recall@K, NDCG@K).